# Задание 1 (5 балла)

Имплементируйте алгоритм Леска (описание есть в семинаре) и оцените качество его работы на датасете `data/corpus_wsd_50k.txt`

В качестве метрики близости вы должны попробовать два подхода:

1) Jaccard score на множествах слов (определений и контекста)
2) Cosine distance на эмбедингах sentence_transformers

В качестве метрики используйте accuracy (% правильных ответов). Предсказывайте только многозначные слова в датасете

Контекст вы можете определить самостоятельно (окно вокруг целевого слова или все предложение). Также можете поэкспериментировать с предобработкой для обоих методов.

In [21]:
import os, re, random, time, math, warnings
from collections import Counter, defaultdict
import numpy as np

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

try:
    from sentence_transformers import SentenceTransformer
    ST_AVAILABLE = True
    print("sentence_transformers доступен.")
except Exception as e:
    ST_AVAILABLE = False
    print("sentence_transformers НЕ доступен:", e)

sentence_transformers доступен.


In [ ]:
CORPUS_PATH = "corpus_wsd_50k.txt"

if not os.path.exists(CORPUS_PATH):
    alt = os.path.join("data", "corpus_wsd_50k.txt")
    if os.path.exists(alt):
        CORPUS_PATH = alt
    else:
        raise FileNotFoundError(
            "Не найден ни 'corpus_wsd_50k.txt', ни 'data/corpus_wsd_50k.txt'. "
        )

with open(CORPUS_PATH, encoding="utf-8") as f:
    _ = f.readline() 

print(f"Файл: {CORPUS_PATH}")

Файл: corpus_wsd_50k.txt


In [ ]:
def parse_corpus(path):
    
    sentences = []
    current = []
    with open(path, encoding="utf-8") as f:
        for raw in f:
            line = raw.rstrip("\n")
            if not line.strip():
                if current:
                    sentences.append(current)
                    current = []
                continue

            parts = line.split("\t")
            if len(parts) >= 3:
                sense_col, lemma, form = parts[0], parts[1], parts[2]
            elif len(parts) == 2:
                sense_col, lemma, form = "", parts[0], parts[1]
            else:
                continue

            sense_col = sense_col.strip()
            lemma = lemma.strip()
            form = form.strip()

            if sense_col and "%" in sense_col:
                lemma_ann, _, sense = sense_col.partition("%")
                current.append({
                    "lemma": lemma_ann.strip(),
                    "sense": sense,
                    "form": form,
                })
            else:
                current.append({
                    "lemma": lemma or None,
                    "sense": None,
                    "form": form,
                })
        if current:
            sentences.append(current)
    return sentences


sentences = parse_corpus(CORPUS_PATH)
print(f"Предложений: {len(sentences):,}")
print("Пример первых 10 токенов первого предложения:")
for t in sentences[0][:10]:
    print("  ", t)

Предложений: 49,452
Пример первых 10 токенов первого предложения:
   {'lemma': 'how', 'sense': None, 'form': 'How'}
   {'lemma': 'long', 'sense': '3:00:02::', 'form': 'long'}
   {'lemma': 'have', 'sense': None, 'form': 'has'}
   {'lemma': 'it', 'sense': None, 'form': 'it'}
   {'lemma': 'be', 'sense': '2:42:03::', 'form': 'been'}
   {'lemma': 'since', 'sense': None, 'form': 'since'}
   {'lemma': 'you', 'sense': None, 'form': 'you'}
   {'lemma': 'review', 'sense': '2:31:00::', 'form': 'reviewed'}
   {'lemma': 'the', 'sense': None, 'form': 'the'}
   {'lemma': 'objective', 'sense': '1:09:00::', 'form': 'objectives'}


In [24]:
examples = []
for sent in sentences:
    forms = [t["form"].lower() for t in sent]
    for i, t in enumerate(sent):
        if t["sense"] is not None:
            ctx_tokens = forms[:i] + forms[i+1:]
            examples.append((t["lemma"], t["sense"], " ".join(ctx_tokens)))

print(f"Всего размеченных примеров: {len(examples):,}")

by_word = defaultdict(lambda: defaultdict(list))
for lemma, sense, ctx in examples:
    by_word[lemma][sense].append(ctx)

polysemous = {w: s for w, s in by_word.items() if len(s) >= 2}
print(f"Многозначных слов: {len(polysemous)}")

top = sorted(polysemous.items(), key=lambda kv: len(kv[1]), reverse=True)[:10]
for w, senses in top:
    print(f"  {w}: {len(senses)} смыслов, "
          f"примеров всего {sum(len(v) for v in senses.values())}")

Всего размеченных примеров: 239,913
Многозначных слов: 5435
  run: 36 смыслов, примеров всего 266
  take: 35 смыслов, примеров всего 745
  make: 31 смыслов, примеров всего 1524
  give: 30 смыслов, примеров всего 802
  play: 30 смыслов, примеров всего 311
  pass: 27 смыслов, примеров всего 204
  break: 27 смыслов, примеров всего 97
  line: 27 смыслов, примеров всего 241
  place: 27 смыслов, примеров всего 387
  hold: 27 смыслов, примеров всего 357


In [25]:
random.seed(SEED)

TRAIN_RATIO = 0.7
train = defaultdict(lambda: defaultdict(list))
test = []

for word, senses in polysemous.items():
    for sid, contexts in senses.items():
        random.shuffle(contexts)
        n_train = max(1, int(len(contexts) * TRAIN_RATIO))
        train[word][sid] = contexts[:n_train]
        for ctx in contexts[n_train:]:
            test.append((word, sid, ctx))

print(f"Тестовых примеров: {len(test):,}")

Тестовых примеров: 59,428


In [26]:
STOPWORDS = set("""
и в во не что он на я с со как а то все она так его но да ты к у же вы за бы по
только ее мне было вот от меня еще нет о из ему теперь когда даже ну вдруг ли
если уже или ни быть был него до вас нибудь опять уж вам ведь там потом себя
ничего ей может они тут где есть надо ней для мы тебя их чем была сам чтоб без
будто чего раз тоже себе под будет ж тогда кто этот того потому этого какой
совсем ним здесь этом один почти мой тем чтобы нее сейчас были куда зачем
всех никогда можно при наконец два об другой хоть после над больше тот через
эти нас про всего них какая много разве три эту моя впрочем хорошо свою этой
перед иногда лучше чуть том нельзя такой им более всегда конечно всю между
""".split())

_tok_re = re.compile(r"[а-яёa-z]+", re.IGNORECASE)

def preprocess(text, remove_stopwords=True):
    tokens = _tok_re.findall(text.lower())
    if remove_stopwords:
        tokens = [t for t in tokens if t not in STOPWORDS and len(t) > 2]
    return tokens

In [ ]:
def jaccard(a, b):
    sa, sb = set(a), set(b)
    if not sa or not sb:
        return 0.0
    return len(sa & sb) / len(sa | sb)


def lesk_jaccard(word, context_tokens, train_data, window=None):
    
    if window is not None:
        ctx_low = [t.lower() for t in context_tokens]
        idx = None
        for i, t in enumerate(ctx_low):
            if t == word.lower():
                idx = i
                break
        if idx is not None:
            context_tokens = context_tokens[max(0, idx - window): idx + window + 1]

    best_sense, best_score = None, -1.0
    for sid, contexts in train_data[word].items():
        gloss_tokens = []
        for ctx in contexts:
            gloss_tokens.extend(preprocess(ctx))
        score = jaccard(gloss_tokens, context_tokens)
        if score > best_score:
            best_score = score
            best_sense = sid
    return best_sense

In [28]:
if ST_AVAILABLE:
    _st_model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
    def embed(texts):
        return _st_model.encode(texts, convert_to_numpy=True, show_progress_bar=False)
else:
    from sklearn.feature_extraction.text import TfidfVectorizer
    _vec = None
    def embed(texts):
        global _vec
        if _vec is None:
            _vec = TfidfVectorizer(min_df=1)
            return _vec.fit_transform(texts).toarray()
        return _vec.transform(texts).toarray()


def cosine(a, b):
    na, nb = np.linalg.norm(a), np.linalg.norm(b)
    if na == 0 or nb == 0:
        return 0.0
    return float(np.dot(a, b) / (na * nb))


def lesk_embed(word, context, train_data):
    ctx_emb = embed([context])[0]
    best_sense, best_score = None, -1.0
    for sid, contexts in train_data[word].items():
        gloss_text = " ".join(contexts)
        gloss_emb = embed([gloss_text])[0]
        score = cosine(ctx_emb, gloss_emb)
        if score > best_score:
            best_score = score
            best_sense = sid
    return best_sense

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1708.22it/s]


In [29]:
def evaluate(method, test_data, train_data, **kwargs):
    correct = 0
    total = 0
    for word, true_sid, ctx in test_data:
        if method == "jaccard":
            toks = preprocess(ctx)
            pred = lesk_jaccard(word, toks, train_data, window=kwargs.get("window"))
        elif method == "embed":
            pred = lesk_embed(word, ctx, train_data)
        else:
            raise ValueError(method)
        total += 1
        if pred == true_sid:
            correct += 1
    return correct / total if total else 0.0

In [31]:
MAX_TEST = 500
random.shuffle(test)
test_small = test[:MAX_TEST]
print(f"Тестируем на {len(test_small)} примерах")

for window in [None, 10, 5]:
    t0 = time.time()
    acc = evaluate("jaccard", test_small, train, window=window)
    dt = time.time() - t0
    print(f"Jaccard, window={window}: acc={acc:.3f}, time={dt:.1f}s")

t0 = time.time()
acc = evaluate("embed", test_small, train)
dt = time.time() - t0
label = "Embeddings" if ST_AVAILABLE else "TF-IDF cosine (fallback)"
print(f"{label}: acc={acc:.3f}, time={dt:.1f}s")

Тестируем на 500 примерах
Jaccard, window=None: acc=0.186, time=3.9s
Jaccard, window=10: acc=0.182, time=4.0s
Jaccard, window=5: acc=0.176, time=4.0s
Embeddings: acc=0.376, time=267.5s


# Задание 2 (5 балла)
Попробуйте разные алгоритмы кластеризации на датасете - `https://github.com/nlpub/russe-wsi-kit/blob/initial/data/main/wiki-wiki/train.csv`

Используйте код из семинара как основу. Используйте ARI как метрику качества.

Попробуйте все 4 алгоритма кластеризации, про которые говорилось на семинаре. Для каждого из алгоритмов попробуйте настраивать гиперпараметры (посмотрите их в документации). Прогоните как минимум 5 экспериментов (не обязательно успешных) с разными параметрами на каждый алгоритме кластеризации и оцените: качество кластеризации, скорость работы, интуитивность параметров.

Помимо этого также выберите 1 дополнительный алгоритм кластеризации отсюда - https://scikit-learn.org/stable/modules/clustering.html , опишите своими словами принцип его работы  и проделайте аналогичные эксперименты. 

In [32]:
import os, re, csv, time, urllib.request, warnings
from collections import defaultdict
import numpy as np

warnings.filterwarnings("ignore")

SEED = 42
np.random.seed(SEED)

try:
    from sentence_transformers import SentenceTransformer
    ST_AVAILABLE = True
    print("sentence_transformers доступен.")
except Exception as e:
    ST_AVAILABLE = False
    print("sentence_transformers НЕ доступен:", e)

from sklearn.cluster import (
    KMeans, AgglomerativeClustering, DBSCAN,
    SpectralClustering, BisectingKMeans
)
from sklearn.metrics import adjusted_rand_score

sentence_transformers доступен.


In [33]:
URL = ("https://raw.githubusercontent.com/nlpub/russe-wsi-kit/"
       "master/data/main/wiki-wiki/train.csv")
CSV_PATH = "wiki-wiki_train.csv"

if not os.path.exists(CSV_PATH):
    print("Скачиваем train.csv...")
    urllib.request.urlretrieve(URL, CSV_PATH)

print(f"Файл: {CSV_PATH}")

Скачиваем train.csv...
Файл: wiki-wiki_train.csv


In [34]:
data = defaultdict(lambda: {"contexts": [], "labels": []})

with open(CSV_PATH, encoding="utf-8") as f:
    reader = csv.reader(f, delimiter="\t")
    for row in reader:
        if len(row) < 6:
            continue
        # первые 5 полей фиксированы, остальное — контекст
        word = row[1].strip()
        gold = row[2].strip()
        context = "\t".join(row[5:]).strip()
        if not word or not gold:
            continue
        data[word]["contexts"].append(context)
        data[word]["labels"].append(gold)

print(f"Слов: {len(data)}")
for w, d in list(data.items())[:10]:
    print(f"  {w}: {len(d['contexts'])} примеров, "
          f"уникальных значений: {len(set(d['labels']))}")

Слов: 5
  word: 1 примеров, уникальных значений: 1
  замок: 138 примеров, уникальных значений: 2
  лук: 110 примеров, уникальных значений: 2
  суда: 135 примеров, уникальных значений: 2
  бор: 56 примеров, уникальных значений: 2


In [35]:
if ST_AVAILABLE:
    model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
    def encode(texts):
        return model.encode(texts, convert_to_numpy=True, show_progress_bar=False)
else:
    from sklearn.feature_extraction.text import TfidfVectorizer
    _vec = None
    def encode(texts):
        global _vec
        if _vec is None:
            _vec = TfidfVectorizer(max_features=2000)
            return _vec.fit_transform(texts).toarray()
        return _vec.transform(texts).toarray()

EMB = {}
for w, d in data.items():
    EMB[w] = encode(d["contexts"])
    print(f"  {w}: {EMB[w].shape}")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2638.75it/s]


  word: (1, 384)
  замок: (138, 384)
  лук: (110, 384)
  суда: (135, 384)
  бор: (56, 384)


In [40]:
def run_experiment(word, algo_name, **params):
    X = EMB[word]
    n_samples = X.shape[0]
    n_true = len(set(data[word]["labels"]))

    if algo_name == "kmeans":
        n_clusters = params.get("n_clusters", n_true)
        if n_clusters > n_samples:
            return None
        algo = KMeans(
            n_clusters=n_clusters,
            init=params.get("init", "k-means++"),
            n_init=params.get("n_init", 10),
            random_state=SEED,
        )
    elif algo_name == "agglo":
        n_clusters = params.get("n_clusters", n_true)
        if n_clusters > n_samples:
            return None
        algo = AgglomerativeClustering(
            n_clusters=n_clusters,
            linkage=params.get("linkage", "ward"),
        )
    elif algo_name == "dbscan":
        algo = DBSCAN(
            eps=params.get("eps", 0.5),
            min_samples=params.get("min_samples", 3),
            metric=params.get("metric", "cosine"),
        )
    elif algo_name == "spectral":
        n_clusters = params.get("n_clusters", n_true)
        if n_clusters > n_samples:
            return None
        algo = SpectralClustering(
            n_clusters=n_clusters,
            affinity=params.get("affinity", "nearest_neighbors"),
            n_neighbors=params.get("n_neighbors", 10),
            assign_labels=params.get("assign_labels", "kmeans"),
            random_state=SEED,
        )
    elif algo_name == "bisecting":
        n_clusters = params.get("n_clusters", n_true)
        if n_clusters > n_samples:
            return None
        algo = BisectingKMeans(
            n_clusters=n_clusters,
            bisecting_strategy=params.get("bisecting_strategy", "biggest_inertia"),
            random_state=SEED,
        )
    else:
        raise ValueError(algo_name)

    try:
        t0 = time.time()
        pred = algo.fit_predict(X)
        dt = time.time() - t0
    except Exception as e:
        return None

    ari = adjusted_rand_score(data[word]["labels"], pred)
    n_clusters_found = len(set(pred) - {-1})
    return {
        "word": word,
        "algo": algo_name,
        "params": params,
        "ari": ari,
        "time": dt,
        "n_clusters": n_clusters_found,
    }

In [41]:
MIN_SAMPLES = 10 
MIN_CLASSES = 2  

valid_words = [
    w for w, d in data.items()
    if len(d["contexts"]) >= MIN_SAMPLES
    and len(set(d["labels"])) >= MIN_CLASSES
]
print(f"Валидных слов: {len(valid_words)} из {len(data)}")

WORDS = valid_words[:3] 
print(f"Слова для экспериментов: {WORDS}")

results = []

for w in WORDS:
    n_true = len(set(data[w]["labels"]))
    n_samples = len(data[w]["contexts"])
    for n_clusters in [n_true, n_true + 1, max(2, n_true - 1), 2, 5]:
        if n_clusters > n_samples:
            continue
        for init in ["k-means++", "random"]:
            r = run_experiment(w, "kmeans", n_clusters=n_clusters, init=init)
            if r is not None:
                results.append(r)

for w in WORDS:
    n_true = len(set(data[w]["labels"]))
    n_samples = len(data[w]["contexts"])
    for linkage in ["ward", "complete", "average", "single"]:
        for n_clusters in [n_true, n_true + 1]:
            if n_clusters > n_samples:
                continue
            r = run_experiment(w, "agglo", n_clusters=n_clusters, linkage=linkage)
            if r is not None:
                results.append(r)

for w in WORDS:
    for eps in [0.3, 0.5, 0.7, 1.0, 1.5]:
        r = run_experiment(w, "dbscan", eps=eps, min_samples=3, metric="cosine")
        if r is not None:
            results.append(r)

for w in WORDS:
    n_true = len(set(data[w]["labels"]))
    n_samples = len(data[w]["contexts"])
    for aff in ["nearest_neighbors", "rbf"]:
        for nl in [n_true, n_true + 1, 2]:
            if nl > n_samples:
                continue
            r = run_experiment(w, "spectral", n_clusters=nl,
                               affinity=aff, n_neighbors=10)
            if r is not None:
                results.append(r)

for w in WORDS:
    n_true = len(set(data[w]["labels"]))
    n_samples = len(data[w]["contexts"])
    for n_clusters in [n_true, n_true + 1, 2, 5]:
        if n_clusters > n_samples:
            continue
        for strat in ["biggest_inertia", "largest_cluster"]:
            r = run_experiment(w, "bisecting",
                               n_clusters=n_clusters,
                               bisecting_strategy=strat)
            if r is not None:
                results.append(r)

print(f"\nВсего успешных экспериментов: {len(results)}")

Валидных слов: 4 из 5
Слова для экспериментов: ['замок', 'лук', 'суда']

Всего успешных экспериментов: 111


In [42]:
import statistics
from collections import defaultdict

by_algo = defaultdict(list)
for r in results:
    by_algo[r["algo"]].append(r)

print(f"{'Algo':<12} {'mean ARI':>9} {'best ARI':>9} "
      f"{'mean time':>10} {'median time':>12}")
for algo, rs in by_algo.items():
    aris = [r["ari"] for r in rs]
    times = [r["time"] for r in rs]
    print(f"{algo:<12} {statistics.mean(aris):>9.3f} {max(aris):>9.3f} "
          f"{statistics.mean(times):>10.3f} {statistics.median(times):>12.3f}")

print("\nЛучшие конфигурации по алгоритмам:")
for algo, rs in by_algo.items():
    best = max(rs, key=lambda r: r["ari"])
    print(f"  {algo}: ARI={best['ari']:.3f}, "
          f"params={best['params']}, word={best['word']}")

Algo          mean ARI  best ARI  mean time  median time
kmeans           0.649     1.000      0.035        0.033
agglo            0.333     1.000      0.492        0.002
dbscan           0.099     0.918      0.002        0.001
spectral         0.355     1.000      0.046        0.034
bisecting        0.581     1.000      0.008        0.006

Лучшие конфигурации по алгоритмам:
  kmeans: ARI=1.000, params={'n_clusters': 2, 'init': 'k-means++'}, word=суда
  agglo: ARI=1.000, params={'n_clusters': 2, 'linkage': 'ward'}, word=суда
  dbscan: ARI=0.918, params={'eps': 0.5, 'min_samples': 3, 'metric': 'cosine'}, word=суда
  spectral: ARI=1.000, params={'n_clusters': 2, 'affinity': 'nearest_neighbors', 'n_neighbors': 10}, word=суда
  bisecting: ARI=1.000, params={'n_clusters': 2, 'bisecting_strategy': 'biggest_inertia'}, word=суда


В качестве дополнительного алгоритма кластеризации из документации scikit-learn я выбрала BisectingKMeans - иерархический вариант обычного KMeans, основанный на стратегии нисходящей кластеризации. Идея алгоритма состоит в том, что вместо одновременного поиска всех центроидов, как это делает классический KMeans, мы изначально полагаем, что весь корпус точек образует один большой кластер, а затем последовательно делим его на части. На каждом шаге выбирается один кластер-кандидат (либо самый крупный по числу точек, либо тот, у которого наибольшая сумма квадратов отклонений от центроида - это регулируется параметром bisecting_strategy), и внутри него запускается обычный KMeans с k=2. Получившиеся два подкластера заменяют исходный в общем списке, и так продолжается до тех пор, пока число кластеров не достигнет заданного значения n_clusters. Это строит бинарное дерево разбиений, где каждый узел представляет собой подмножество точек, а листья соответствуют финальным кластерам. Такая схема даёт несколько преимуществ: во-первых, алгоритм работает быстрее классического KMeans при большом числе кластеров, потому что на каждой итерации он имеет дело не со всем корпусом, а лишь с одним подмножеством, что особенно заметно на данных вроде WSI, где n_clusters может достигать десятков. Во-вторых, поскольку KMeans с k=2 почти всегда сходится к осмысленному разбиению, алгоритм практически никогда не оставляет пустых кластеров, в отличие от обычного KMeans, который при неудачной инициализации может «потерять» один из центроидов. В-третьих, результат получается иерархически упорядоченным: кластеры, которые отделились раньше, как правило, сильнее отличаются друг от друга, а более поздние разбиения уточняют структуру внутри уже сформировавшихся групп. Главный недостаток оттуда же: поскольку центроиды выбираются не одновременно, а поочерёдно, алгоритм не может воспользоваться преимуществами инициализации k-means++, и качество финального разбиения оказывается в среднем чуть ниже, чем у классического KMeans с той же схемой инициализации. Кроме того, жадность решений на каждом шаге означает, что ошибка, допущенная при раннем делении, никак не исправляется позже, и это иногда приводит к неоптимальной иерархии. В контексте задачи определения значений слов это выражается в том, что BisectingKMeans обычно показывает сопоставимое с KMeans значение ARI, но чуть более стабильное: он реже «схлопывает» редкое значение в доминирующий кластер, что для WSI важно, поскольку именно редкие смыслы чаще всего теряются. Параметры у него интуитивны: n_clusters задаёт желаемое число значений, bisecting_strategy отвечает за то, какой кластер дробить следующим, а random_state фиксирует воспроизводимость. Нет eps, bandwidth или affinity, как у DBSCAN или Spectral, поэтому настройка алгоритма и интерпретация его результатов оказываются проще, чем у большинства других методов.